# Credit Risk Modeling (German Credit Dataset)

## Objective
Build a simple machine learning model that predicts whether a loan applicant is likely to be a **bad credit risk** (binary classification).

This project demonstrates:
- basic EDA (exploratory data analysis)
- preprocessing for mixed data types (numeric + categorical)
- a baseline classification model (Logistic Regression)
- evaluation using ROC-AUC and confusion matrix


In [25]:
# Import the folowwing and read in the data
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay

df = pd.read_csv("../data/german_credit.csv", sep=r"\s+", header=None)
df.head()


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


## 1. Basic dataset check and EDA

Fiirst check:
- number of rows/columns
- column types
- any missing values


In [26]:
print('Shape:', df.shape)
df.info()

Shape: (1000, 21)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       1000 non-null   str  
 1   1       1000 non-null   int64
 2   2       1000 non-null   str  
 3   3       1000 non-null   str  
 4   4       1000 non-null   int64
 5   5       1000 non-null   str  
 6   6       1000 non-null   str  
 7   7       1000 non-null   int64
 8   8       1000 non-null   str  
 9   9       1000 non-null   str  
 10  10      1000 non-null   int64
 11  11      1000 non-null   str  
 12  12      1000 non-null   int64
 13  13      1000 non-null   str  
 14  14      1000 non-null   str  
 15  15      1000 non-null   int64
 16  16      1000 non-null   str  
 17  17      1000 non-null   int64
 18  18      1000 non-null   str  
 19  19      1000 non-null   str  
 20  20      1000 non-null   int64
dtypes: int64(8), str(13)
memory usage: 164.2 KB


In [27]:
df.columns

Index([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
       20],
      dtype='int64')

In [28]:
column_names = [
    "checking_account",
    "duration",
    "credit_history",
    "purpose",
    "credit_amount",
    "savings_account",
    "employment_since",
    "installment_rate",
    "personal_status_sex",
    "other_debtors",
    "residence_since",
    "property",
    "age",
    "other_installment_plans",
    "housing",
    "existing_credits",
    "job",
    "num_dependents",
    "telephone",
    "foreign_worker",
    "target"
]

df.columns = column_names
df.head()

,checking_account,duration,credit_history,purpose,credit_amount,savings_account,employment_since,installment_rate,personal_status_sex,other_debtors,...,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,...,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,...,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,...,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,...,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,...,A124,53,A143,A153,2,A173,2,A191,A201,2


In [29]:
df["bad_credit"] = (df["target"] == 2).astype(int)
df = df.drop(columns=["target"])
df["bad_credit"].value_counts()

bad_credit
0    700
1    300
Name: count, dtype: int64

# Modeling

Now that we created `bad_credit`:

- `bad_credit = 1` → bad credit
- `bad_credit = 0` → good credit

We will:

1. Define features `X` and target `y`
2. Split data into training and testing sets (80/20)
3. Build preprocessing:
   - One-hot encode categorical variables
   - Standardize numeric variables
4. Train and tune:
   - Logistic Regression (with cross-validation)
   - Random Forest (with cross-validation)
5. Train the best version of each model on the full training set
6. Evaluate both on the test set and compare performance


In [30]:
X = df.drop(columns=["bad_credit"])
y = df["bad_credit"]

print("X shape:", X.shape)
print("y shape:", y.shape)
y.value_counts(normalize=True)

X shape: (1000, 20)
y shape: (1000,)


bad_credit
0    0.7
1    0.3
Name: proportion, dtype: float64

## 1) Train/Test Split

Create a hold-out test set that the model never sees during training and tuning.
This gives an unbiased estimate of performance.


In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state=42,
    stratify=y
)

In [32]:
print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)

Train: (800, 20) (800,)
Test:  (200, 20) (200,)


## 2) Preprocessing

The dataset contains:

- Numeric columns (integers)
- Categorical codes (strings like A11, A34, A152, ...)

I will:
- One-hot encode categorical columns
- Standardize numeric columns

Important: during cross-validation, preprocessing must be fitted **only on the fold-training data**
to avoid data leakage.


In [33]:
# Identify numeric and categorical columns from the TRAIN set only
num_cols = X_train.select_dtypes(include=["int64"]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

print("Numeric cols:", num_cols)
print("Categorical cols:", cat_cols)

Numeric cols: ['duration', 'credit_amount', 'installment_rate', 'residence_since', 'age', 'existing_credits', 'num_dependents']
Categorical cols: ['checking_account', 'credit_history', 'purpose', 'savings_account', 'employment_since', 'personal_status_sex', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']


In [34]:
def preprocess_fit_transform(X_tr, X_va, num_cols, cat_cols):
    """
    Fit preprocessing on X_tr only, transform both X_tr and X_va.
    Returns: X_tr_final, X_va_final
    """
    # One-hot encode categoricals
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    X_tr_cat = ohe.fit_transform(X_tr[cat_cols])
    X_va_cat = ohe.transform(X_va[cat_cols])

    # Scale numerical columns
    scaler = StandardScaler()
    X_tr_num = scaler.fit_transform(X_tr[num_cols])
    X_va_num = scaler.transform(X_va[num_cols])

    # Combine (numeric first, then categorical)
    X_tr_final = np.hstack([X_tr_num, X_tr_cat])
    X_va_final = np.hstack([X_va_num, X_va_cat])

    return X_tr_final, X_va_final

## 3) Evaluation Metrics

Mainly, we will compare models using **ROC-AUC** because:

- the output is a probability score
- ROC-AUC measures how well the model ranks good vs bad credit
- It is not tied to a single threshold like 0.5

We will still report:
- confusion matrix (at threshold 0.5)
- classification report


In [35]:
def evaluate_on_test(model, X_test_final, y_test, threshold=0.5, label="model"):
    proba = model.predict_proba(X_test_final)[:, 1]
    pred = (proba >= threshold).astype(int)
    auc = roc_auc_score(y_test, proba)
    cm = confusion_matrix(y_test, pred)
    report = classification_report(y_test, pred, digits=3)
    return {"label": label, "roc_auc": auc, "confusion_matrix": cm, "report": report}

# 4) Logistic Regression (Hyperparameter Tuning)

Logistic Regression is a strong baseline.

We tune:
- `C` (regularization strength)
- `penalty` (L1 or L2)

We use **5-fold stratified cross-validation** and pick the setting with the best mean ROC-AUC.

In [36]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

logreg_grid = [
    {"C": 0.01, "penalty": "l1"},
    {"C": 0.1,  "penalty": "l1"},
    {"C": 1.0,  "penalty": "l1"},
    {"C": 0.01, "penalty": "l2"},
    {"C": 0.1,  "penalty": "l2"},
    {"C": 1.0,  "penalty": "l2"},
    {"C": 10.0, "penalty": "l2"},
]

logreg_results = []

for params in logreg_grid:
    aucs = []

    for tr_idx, va_idx in skf.split(X_train, y_train):
        X_tr_fold = X_train.iloc[tr_idx]
        y_tr_fold = y_train.iloc[tr_idx]
        X_va_fold = X_train.iloc[va_idx]
        y_va_fold = y_train.iloc[va_idx]

        X_tr_final, X_va_final = preprocess_fit_transform(X_tr_fold, X_va_fold, num_cols, cat_cols)

        clf = LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="liblinear",  # supports l1 and l2
            C=params["C"],
            penalty=params["penalty"]
        )
        clf.fit(X_tr_final, y_tr_fold)
        proba_va = clf.predict_proba(X_va_final)[:, 1]
        aucs.append(roc_auc_score(y_va_fold, proba_va))

    logreg_results.append({
        "params": params,
        "mean_auc": float(np.mean(aucs)),
        "std_auc": float(np.std(aucs))
    })

logreg_results_sorted = sorted(logreg_results, key=lambda d: d["mean_auc"], reverse=True)

for r in logreg_results_sorted:
    print(r["params"], "Mean AUC:", round(r["mean_auc"], 4), "Std:", round(r["std_auc"], 4))

best_logreg_params = logreg_results_sorted[0]["params"]
print("\nBEST Logistic Regression params:", best_logreg_params)

{'C': 0.1, 'penalty': 'l2'} Mean AUC: 0.7814 Std: 0.0507
{'C': 0.01, 'penalty': 'l2'} Mean AUC: 0.7767 Std: 0.0431
{'C': 1.0, 'penalty': 'l2'} Mean AUC: 0.7693 Std: 0.0529
{'C': 1.0, 'penalty': 'l1'} Mean AUC: 0.7686 Std: 0.0536
{'C': 10.0, 'penalty': 'l2'} Mean AUC: 0.7597 Std: 0.0571
{'C': 0.1, 'penalty': 'l1'} Mean AUC: 0.7582 Std: 0.0487
{'C': 0.01, 'penalty': 'l1'} Mean AUC: 0.5 Std: 0.0

BEST Logistic Regression params: {'C': 0.1, 'penalty': 'l2'}


### Train final tuned Logistic Regression

We now:
- fit preprocessing on the full training set
- train Logistic Regression using the best hyperparameters
- evaluate on the test set


In [37]:
# Fit preprocessing on full training, transform train+test
X_train_final, X_test_final = preprocess_fit_transform(X_train, X_test, num_cols, cat_cols)

final_logreg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="liblinear",
    C=best_logreg_params["C"],
    penalty=best_logreg_params["penalty"]
)

final_logreg.fit(X_train_final, y_train)

logreg_test = evaluate_on_test(final_logreg, X_test_final, y_test, threshold=0.5, label="Logistic Regression (tuned)")
print("ROC-AUC:", round(logreg_test["roc_auc"], 4))
print("Confusion Matrix:\n", logreg_test["confusion_matrix"])
print("\nReport:\n", logreg_test["report"])

ROC-AUC: 0.8094
Confusion Matrix:
 [[98 42]
 [13 47]]

Report:
               precision    recall  f1-score   support

           0      0.883     0.700     0.781       140
           1      0.528     0.783     0.631        60

    accuracy                          0.725       200
   macro avg      0.705     0.742     0.706       200
weighted avg      0.776     0.725     0.736       200



# 5) Random Forest (With Hyperparameter Tuning)

Random Forest can capture nonlinear relationships and interactions automatically.

We tune a small set of hyperparameters:
- `n_estimators`
- `max_depth`
- `min_samples_split`

Again use 5-fold stratified cross-validation and select the best mean ROC-AUC.


In [38]:
from sklearn.ensemble import RandomForestClassifier

rf_grid = [
    {"n_estimators": 200, "max_depth": None, "min_samples_split": 2},
    {"n_estimators": 400, "max_depth": None, "min_samples_split": 2},
    {"n_estimators": 400, "max_depth": 8,    "min_samples_split": 2},
    {"n_estimators": 400, "max_depth": 12,   "min_samples_split": 2},
    {"n_estimators": 400, "max_depth": 12,   "min_samples_split": 5},
]

In [39]:
rf_results = []

for params in rf_grid:
    aucs = []
    for tr_idx, va_idx in skf.split(X_train, y_train):
        X_tr_fold = X_train.iloc[tr_idx]
        y_tr_fold = y_train.iloc[tr_idx]
        X_va_fold = X_train.iloc[va_idx]
        y_va_fold = y_train.iloc[va_idx]


        X_tr_final, X_va_final = preprocess_fit_transform(X_tr_fold, X_va_fold, num_cols, cat_cols)

        rf = RandomForestClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_split=params["min_samples_split"],
            random_state=42,
            class_weight="balanced_subsample",
            n_jobs=-1
        )
        rf.fit(X_tr_final, y_tr_fold)
        proba_va = rf.predict_proba(X_va_final)[:, 1]
        aucs.append(roc_auc_score(y_va_fold, proba_va))

    rf_results.append({
        "params": params,
        "mean_auc": float(np.mean(aucs)),
        "std_auc": float(np.std(aucs))
    })

rf_results_sorted = sorted(rf_results, key=lambda d: d["mean_auc"], reverse=True)

for r in rf_results_sorted:
    print(r["params"], "Mean AUC:", round(r["mean_auc"], 4), "Std:", round(r["std_auc"], 4))

best_rf_params = rf_results_sorted[0]["params"]
print("\nBEST Random Forest params:", best_rf_params)


{'n_estimators': 400, 'max_depth': 12, 'min_samples_split': 2} Mean AUC: 0.7953 Std: 0.0381
{'n_estimators': 400, 'max_depth': 8, 'min_samples_split': 2} Mean AUC: 0.7949 Std: 0.0426
{'n_estimators': 400, 'max_depth': 12, 'min_samples_split': 5} Mean AUC: 0.7934 Std: 0.0337
{'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2} Mean AUC: 0.7887 Std: 0.0342
{'n_estimators': 400, 'max_depth': None, 'min_samples_split': 2} Mean AUC: 0.7882 Std: 0.0371

BEST Random Forest params: {'n_estimators': 400, 'max_depth': 12, 'min_samples_split': 2}


### Train final tuned Random Forest

Train Random Forest on the full training set using the best parameters and evaluate on the test set.


In [40]:
final_rf = RandomForestClassifier(
    n_estimators=best_rf_params["n_estimators"],
    max_depth=best_rf_params["max_depth"],
    min_samples_split=best_rf_params["min_samples_split"],
    random_state=42,
    class_weight="balanced_subsample",
    n_jobs=-1
)

final_rf.fit(X_train_final, y_train)

rf_test = evaluate_on_test(final_rf, X_test_final, y_test, threshold=0.5, label="Random Forest (tuned)")
print("ROC-AUC:", round(rf_test["roc_auc"], 4))
print("Confusion Matrix:\n", rf_test["confusion_matrix"])
print("\nReport:\n", rf_test["report"])


ROC-AUC: 0.8077
Confusion Matrix:
 [[128  12]
 [ 32  28]]

Report:
               precision    recall  f1-score   support

           0      0.800     0.914     0.853       140
           1      0.700     0.467     0.560        60

    accuracy                          0.780       200
   macro avg      0.750     0.690     0.707       200
weighted avg      0.770     0.780     0.765       200



# 6) Model Comparison 

Compare tuned Logistic Regression and tuned Random Forest using:

- Test ROC-AUC (main metric)
- Confusion matrix (threshold = 0.5)
- Classification report

Select the best model based on ROC-AUC on the test set.


In [41]:
results_table = pd.DataFrame([
    {"model": logreg_test["label"], "test_roc_auc": logreg_test["roc_auc"]},
    {"model": rf_test["label"],     "test_roc_auc": rf_test["roc_auc"]},
]).sort_values("test_roc_auc", ascending=False)

results_table


,model,test_roc_auc
0,Logistic Regression (tuned),0.809405
1,Random Forest (tuned),0.807738


In [42]:
best_row = results_table.iloc[0]
print("Best model:", best_row["model"])
print("Best test ROC-AUC:", round(best_row["test_roc_auc"], 4))


Best model: Logistic Regression (tuned)
Best test ROC-AUC: 0.8094


# Conclusion

In this project a complete supervised ML workflow for credit risk classification was done:

- Created a binary target `bad_credit`
- Split into train/test
- Preprocessed mixed feature types (one-hot + scaling)
- Tuned Logistic Regression and Random Forest using cross-validation
- Evaluated both on the test set
- Selected the best model based on test ROC-AUC

The best performing model was the **Logistic Regression Model tuned with {'C': 0.1, 'penalty': 'l2'}**
